# Segmental accent + prosody transfer from another voice

1. Text → IPA, one chunk per sentence
2. `manipulate_with_quota()` applies your segmental accent rules to the IPA: everything `manipulate()` does, plus `word_quota` rules that change **exactly N words** per phoneme, one change per word (see `word_quota.py`)
3. The **target** voice speaks the accented IPA (timbre + segments). Its **duration and/or pitch and/or energy** can come from one or several **donor** voices (one donor per sentence, as in the prosody notebook)
4. The donor reads either
   - `donor_text="original"`: the **unmanipulated** text, as a native speaker would say it. Its durations are mapped onto the accented tokens through the `origins` from `manipulate()`, and its pitch/energy are predicted on the original string on a timeline that lines up frame by frame with the accented one
   - `donor_text="manipulated"`: the **accented** text itself, so its timing and melody react to the substituted phonemes

**Palatalization (`kʲ`, `tʲ`, `pʲ` …).** The base consonant takes the donor's duration; the `ʲ` that the rule added keeps the target voice's own duration:

| | original `k` | accented `k` | accented `ʲ` |
|---|---|---|---|
| `synth_aligned(dur_source="original")` | 6 frames | 5 (split) | 1 (split) |
| **here** (`keep_own=("ʲ",)`) | 6 frames (donor) | **6** (donor) | **target's own** (e.g. 2) |

This holds for both `donor_text` settings. Only `ʲ` that a rule *added* is affected (a `ʲ` already in the original string is treated like any other token). Add `"ʰ"` (or `"ː"`) to `keep_own` to treat other added diacritics the same way, or set `keep_own=()` to go back to splitting.

With `dur_mode="rate"`, the `ʲ` frames are held fixed and only the other tokens are scaled to the target's sentence length, so the `k` gets the donor's `k` × the sentence's rate factor. With `dur_mode="raw"` the `k` gets exactly the donor's frames.

In [ ]:
from functions import *          # model, g2p, manipulate, text_to_ipa_chunks, transfer_durations, align_phonemes,
                                 # audiovisualize_interactive, print_stats, check_ipa, VOCAB, SR, FRAME_SAMPLES ...
import functions as F            # to reuse the player's HTML template
from functions import _encode, _voice_pack, _durations_cached   # underscore names are not exported by *
from word_quota import manipulate_with_quota, normalize_ipa   # manipulate() + "word_quota" rules (exactly N words per vowel)
import torch, numpy as np, json, os, uuid
import soundfile as sf
from IPython.display import display, HTML

HOP   = FRAME_SAMPLES // 2       # F0 / energy frame: 300 samples = 12.5 ms
HOP_S = HOP / SR

## Model helpers

In [ ]:
@torch.no_grad()
def make_alignment(ps, dur):
    """Token-to-frame alignment matrix (1, n_tokens, n_frames) from durations."""
    dev = model.device
    n = _encode(ps)["n"]
    dur = torch.as_tensor(dur, device=dev).long().clamp(min=0).reshape(-1)
    assert dur.shape[0] == n, f"durations ({dur.shape[0]}) != tokens incl. BOS/EOS ({n})"
    idx = torch.repeat_interleave(torch.arange(n, device=dev), dur)
    aln = torch.zeros((n, idx.shape[0]), device=dev)
    aln[idx, torch.arange(idx.shape[0], device=dev)] = 1
    return aln.unsqueeze(0)


@torch.no_grad()
def predict_f0n(ps, pack, aln):
    """Predicted F0 (Hz) and energy N of voice `pack` on the given alignment (numpy, 2*n_frames)."""
    enc = _encode(ps)
    s = pack[len(ps) - 1][:, 128:]
    d = model.predictor.text_encoder(enc["d_en"], s, enc["L"], enc["tm"])
    F0, N = model.predictor.F0Ntrain(d.transpose(-1, -2) @ aln, s)
    return F0.squeeze().cpu().numpy(), N.squeeze().cpu().numpy()


@torch.no_grad()
def decode(ps, pack, aln, f0, n):
    """Decoder with voice `pack`'s timbre and the GIVEN F0 / energy curves."""
    enc = _encode(ps)
    ref_s = pack[len(ps) - 1]
    to_t = lambda x: torch.as_tensor(x, dtype=torch.float32, device=model.device).unsqueeze(0)
    return model.decoder(enc["t_en"] @ aln, to_t(f0), to_t(n), ref_s[:, :128]).squeeze().cpu().numpy()

## Normalization helpers

In [ ]:
VOICED_HZ = 50.0

def voiced_mask(f0, thr=VOICED_HZ, rel=0.6):
    """Voiced = above thr AND above rel * the median voiced pitch (drops on/offset frames)."""
    v = f0 > thr
    return v & (f0 > rel * np.median(f0[v])) if v.any() else v


def f0_stats(f0_list):
    """(mean, std) of log-F0 over the voiced frames of several chunks."""
    lf = np.log(np.concatenate([f[voiced_mask(f)] for f in f0_list]))
    return lf.mean(), lf.std()


def en_stats(n_list):
    x = np.concatenate(n_list)
    return x.mean(), x.std()


def apply_f0(f0_d, st_d, st_t, mode):
    """Donor contour into the target's register. Unvoiced frames keep the donor's values."""
    if mode == "raw":
        return f0_d.copy()
    out, v = f0_d.copy(), voiced_mask(f0_d)
    z = np.log(f0_d[v]) - st_d[0]
    if mode == "meanstd":
        z = z / st_d[1] * st_t[1]
    out[v] = np.exp(z + st_t[0])
    return out


def apply_energy(n_d, st_d, st_t, mode):
    if mode == "raw":
        return n_d.copy()
    z = n_d - st_d[0]
    if mode == "meanstd":
        z = z / (st_d[1] + 1e-8) * st_t[1]
    return z + st_t[0]


def scale_durations(dur, factor, total=None):
    """Multiply durations by `factor` and round so the sum is EXACTLY `total`
    (default: round(sum * factor)), with every token >= 1 frame.
    Largest-remainder rounding: frames go to / come from the tokens whose
    rounding error is largest, so each token stays as close as possible to dur*factor."""
    x = np.asarray(dur, float) * factor
    total = int(round(x.sum())) if total is None else int(total)
    assert total >= len(x), f"total {total} < number of tokens {len(x)}"
    out = np.maximum(np.floor(x), 1).astype(int)
    while out.sum() < total:                       # add to the most under-rounded tokens
        k = min(total - out.sum(), len(x))
        out[np.argsort(-(x - out))[:k]] += 1
    while out.sum() > total:                       # remove from the most over-rounded tokens (never below 1)
        cand = np.where(out > 1)[0]
        k = min(out.sum() - total, len(cand))
        out[cand[np.argsort(x[cand] - out[cand])[:k]]] -= 1
    return out.tolist()

## Duration mapping between the original and the accented string
- `map_durations`: donor durations of the **original** string → accented string. Tokens added by a rule that are in `keep_own` (`ʲ`) are taken out, `transfer_durations` maps the rest (so `k` in `kʲ` gets all the frames of the original `k`), and the `ʲ` gets the target's own frames back.
- `back_map_durations`: the opposite direction. Final accented durations → original string, so the donor's pitch/energy can be predicted on the original text with exactly the same number of frames, group by group (the original `k` covers the time of the accented `k` + `ʲ`).
- `rescale_free`: rate normalization that keeps the fixed (`ʲ`) frames and scales only the others.

In [ ]:
def _identity(ps):
    return [(j, j + 1) for j in range(len(ps))]


def own_flags(ps, mask, keep_own=("ʲ",)):
    '''One bool per KEPT token of `ps` (no BOS/EOS): True for a token in `keep_own`
    that a rule added (mask True). These keep the target voice's own duration.'''
    if mask is None or not keep_own:
        return [False] * sum(c in VOCAB for c in ps)
    return [bool(mask[j]) and ps[j] in keep_own for j, c in enumerate(ps) if c in VOCAB]


def map_durations(orig_ps, manip_ps, origin, src_dur, own_dur, own, diacritic_weight=0.3):
    '''src_dur : durations of ORIGINAL string (a donor, or the target), len kept(orig)+2
    own_dur : the target's durations of the ACCENTED string, len kept(manip)+2
    own     : own_flags(manip_ps, ...), one bool per kept accented token
    -> durations of the accented string. own tokens get own_dur, the others share the frames
       of what they replaced (transfer_durations); sum = sum(src_dur) + frames of the own tokens.'''
    origin = _identity(manip_ps) if origin is None else origin
    kept = [j for j, c in enumerate(manip_ps) if c in VOCAB]
    assert len(own) == len(kept) and len(own_dur) == len(kept) + 2
    drop = {j for j, f in zip(kept, own) if f}
    red_ps = "".join(c for j, c in enumerate(manip_ps) if j not in drop)
    red_or = [o for j, o in enumerate(origin) if j not in drop]
    red = transfer_durations(orig_ps, red_ps, red_or, src_dur, diacritic_weight)
    rest = iter(red[1:-1])
    out = [red[0]] + [own_dur[k] if f else next(rest) for k, f in enumerate(own, start=1)] + [red[-1]]
    assert next(rest, None) is None
    return out


def _groups(orig_ps, origin):
    '''Same grouping as transfer_durations: consecutive accented chars with overlapping source
    spans form one group; the groups' source ranges [a, b) partition the original string.'''
    groups = []
    for oi, (a, b) in enumerate(origin):
        if groups and a < groups[-1][1]:
            groups[-1][1] = max(groups[-1][1], b); groups[-1][2].append(oi)
        else:
            groups.append([a, b, [oi]])
    if groups:
        groups[0][0] = 0
        for g, nxt in zip(groups, groups[1:]):
            g[1] = nxt[0]
        groups[-1][1] = len(orig_ps)
    return groups


def _split(total, weights):
    '''Largest-remainder split of `total` frames by `weights`.'''
    ideal = [total * w / sum(weights) for w in weights]
    alloc = [int(v) for v in ideal]
    for j in sorted(range(len(ideal)), key=lambda j: ideal[j] - alloc[j], reverse=True)[:total - sum(alloc)]:
        alloc[j] += 1
    return alloc


def back_map_durations(orig_ps, manip_ps, origin, manip_dur, ref_orig_dur):
    '''Accented durations -> durations of the ORIGINAL string with the SAME total, group by group.
    Inside a group with several original tokens (e.g. tʃ -> ʨ) the frames are split in proportion
    to `ref_orig_dur` (the donor's own durations of the original string).'''
    origin = _identity(manip_ps) if origin is None else origin
    m, k = [0] * len(manip_ps), 1
    for j, c in enumerate(manip_ps):
        if c in VOCAB:
            m[j] = manip_dur[k]; k += 1
    assert k == len(manip_dur) - 1, "manip_dur does not match manip_ps"
    r, k = [0] * len(orig_ps), 1
    for j, c in enumerate(orig_ps):
        if c in VOCAB:
            r[j] = ref_orig_dur[k]; k += 1
    assert k == len(ref_orig_dur) - 1, "ref_orig_dur does not match orig_ps"
    o, carry, last = [0] * len(orig_ps), 0, None
    for a, b, outs in _groups(orig_ps, origin):
        total = sum(m[oi] for oi in outs) + carry
        kept = [j for j in range(a, b) if orig_ps[j] in VOCAB]
        if not kept:                                   # nothing audible in the original here
            carry = total; continue
        carry = 0
        for j, v in zip(kept, _split(total, [max(r[j], 1) for j in kept])):
            o[j] = v
        last = kept[-1]
    if carry and last is not None:
        o[last] += carry
    out = [manip_dur[0]] + [o[j] for j, c in enumerate(orig_ps) if c in VOCAB] + [manip_dur[-1]]
    assert sum(out) == sum(manip_dur), (sum(out), sum(manip_dur))
    return out


def rescale_free(dur, fixed, total):
    '''Scale the non-fixed durations so the whole sentence has `total` frames; fixed ones unchanged.'''
    free = [i for i, f in enumerate(fixed) if not f]
    fsum = sum(d for d, f in zip(dur, fixed) if f)
    x = [dur[i] for i in free]
    new = scale_durations(x, (total - fsum) / sum(x), total=total - fsum)
    out = list(dur)
    for i, v in zip(free, new):
        out[i] = v
    return out

## Synthesis: segmental accent + per-sentence donor prosody
Same return values as `synth_prosody`: `audio, segments, seg_words, seg_altered, pros`.

| argument | meaning |
|---|---|
| `ipa_chunks` | original IPA (what the donor reads with `donor_text="original"`) |
| `manip_chunks, masks, origins` | from `manipulate(..., return_mask=True, return_origin=True)`; leave out for the native version |
| `pitch, energy, duration` | what is transferred from the donor |
| `donor_text` | `"original"` or `"manipulated"`: which text the donor reads |
| `target_timing` | the target's own timing on the accented text (used when `duration=False`, and as the reference length for `dur_mode="rate"`): `"manipulated"` = the target times the accented string itself; `"original"` = the target's timing of the original text mapped onto it (like `SAME_TIMING=True`), with the same `ʲ` rule |
| `keep_own` | added tokens that keep the target's own duration (default `("ʲ",)`) |
| `diacritic_weight` | share given to other added diacritics (not in `keep_own`) when frames are split |
| the rest | as in `synth_prosody` (`f0_mode`, `energy_mode`, `dur_mode`, `norm_scope`, `dur_scope`, `assign`, `seed`) |

In [ ]:
def _voice_name(v, k):
    return v if isinstance(v, str) else f"voice{k}"


def assign_donors(n, donors, assign="balanced", seed=0):
    '''Donor index for each of n sentences.'''
    k = len(donors)
    names = [_voice_name(v, i) for i, v in enumerate(donors)]
    if isinstance(assign, (list, tuple)):
        assert len(assign) == n, f"assign has {len(assign)} entries for {n} sentences"
        return [a if isinstance(a, (int, np.integer)) else names.index(a) for a in assign]
    rng = np.random.default_rng(seed)
    if assign == "random":
        return rng.integers(0, k, n).tolist()
    if assign == "balanced":
        idx = np.concatenate([np.tile(np.arange(k), n // k), rng.choice(k, n % k, replace=False)])
        rng.shuffle(idx)
        return idx.astype(int).tolist()
    raise ValueError(f"unknown assign: {assign!r}")


def synth_accent(ipa_chunks, target_voice, donor_voices=None,
                 manip_chunks=None, masks=None, origins=None,
                 pitch=False, energy=False, duration=False,
                 donor_text="original", target_timing="manipulated",
                 keep_own=("ʲ",), diacritic_weight=0.3,
                 f0_mode="meanstd", energy_mode="meanstd", dur_mode="rate",
                 norm_scope="text", dur_scope="sentence",
                 assign="balanced", seed=0, speed=1.0, sr=SR):
    assert donor_text in ("original", "manipulated"), donor_text
    assert target_timing in ("original", "manipulated"), target_timing
    assert norm_scope in ("text", "sentence") and dur_scope in ("text", "sentence") and dur_mode in ("raw", "rate")
    if manip_chunks is None:                           # native: the accented text IS the original
        manip_chunks, masks, origins = ipa_chunks, None, None
    assert len(manip_chunks) == len(ipa_chunks), "manip_chunks must be parallel to ipa_chunks"
    if donor_voices is None:
        donor_voices = []
    if isinstance(donor_voices, str) or torch.is_tensor(donor_voices):
        donor_voices = [donor_voices]
    transfer = pitch or energy or duration
    assert donor_voices or not transfer, "pitch / energy / duration transfer needs donor_voices"
    keep_own = tuple(keep_own or ())
    names = [_voice_name(v, i) for i, v in enumerate(donor_voices)]
    tpack, tkey = _voice_pack(target_voice)
    dpacks = [_voice_pack(v) for v in donor_voices]                 # [(pack, key)]
    need_origin = (transfer and donor_text == "original") or target_timing == "original"

    # ---- pass 0: sentences, their original / accented strings, donor of each ----
    items = []
    for ci, (orig, ps) in enumerate(zip(ipa_chunks, manip_chunks)):
        if not ps.strip():
            continue
        origin = origins[ci] if origins is not None else (_identity(ps) if ps == orig else None)
        assert origin is not None or not need_origin, \
            f"sentence {ci} was manipulated: pass origins=... from manipulate(..., return_origin=True)"
        mask = masks[ci] if masks is not None else None
        own = own_flags(ps, mask, keep_own)
        items.append(dict(ci=ci, orig=orig, ps=ps, origin=origin, mask=mask, own=own,
                          fixed=[False] + own + [False]))
    dis = assign_donors(len(items), donor_voices, assign, seed) if donor_voices else [None] * len(items)
    for it, di in zip(items, dis):
        it["di"], it["donor"] = di, (names[di] if di is not None else None)

    def own_sum(it):                                   # frames of the tokens that keep the target's duration
        return sum(d for d, f in zip(it["dur_own"], it["fixed"]) if f)

    def donor_dur(it, d):                              # donor d's durations on the accented tokens
        dp, dk = dpacks[d]
        if donor_text == "manipulated":
            src_ = _durations_cached(it["ps"], dp, dk, speed)
            return [o if f else s_ for s_, o, f in zip(src_, it["dur_own"], it["fixed"])]
        return map_durations(it["orig"], it["ps"], it["origin"], _durations_cached(it["orig"], dp, dk, speed),
                             it["dur_own"], it["own"], diacritic_weight)

    # ---- pass 1: durations ----
    for it in items:
        it["dur_own"] = _durations_cached(it["ps"], tpack, tkey, speed)          # target on the accented text
        if target_timing == "manipulated":
            it["dur_t"] = list(it["dur_own"])
        else:
            it["dur_t"] = map_durations(it["orig"], it["ps"], it["origin"],
                                        _durations_cached(it["orig"], tpack, tkey, speed),
                                        it["dur_own"], it["own"], diacritic_weight)
        it["dur_d"] = donor_dur(it, it["di"]) if (duration and it["di"] is not None) else None
    if duration and dur_mode == "rate" and dur_scope == "text":
        # each donor slowed/sped to the target's overall rate (fixed ʲ frames excluded), then the text total is made exact
        free_t = sum(sum(it["dur_t"]) - own_sum(it) for it in items)
        rate = {d: free_t / sum(sum(donor_dur(it, d)) - own_sum(it) for it in items)
                for d in sorted({it["di"] for it in items})}
        L = np.array([(sum(it["dur_d"]) - own_sum(it)) * rate[it["di"]] for it in items])
        L *= free_t / L.sum()
        budgets = np.diff(np.round(np.concatenate([[0], np.cumsum(L)])).astype(int))
    for i, it in enumerate(items):
        if not duration:
            it["dur"] = list(it["dur_t"])
        elif dur_mode == "raw":
            it["dur"] = list(it["dur_d"])
        elif dur_scope == "sentence":
            it["dur"] = rescale_free(it["dur_d"], it["fixed"], sum(it["dur_t"]))
        else:
            it["dur"] = rescale_free(it["dur_d"], it["fixed"], int(budgets[i]) + own_sum(it))

    # ---- pass 2: F0 / energy of the target (accented text) and of EVERY donor (the text it reads) ----
    for it in items:
        it["aln"] = make_alignment(it["ps"], it["dur"])
        it["f0_t"], it["n_t"] = predict_f0n(it["ps"], tpack, it["aln"])
        it["f0_d"] = it["n_d"] = it["d_dur"] = None
        if pitch or energy:
            if donor_text == "manipulated":
                it["d_ps"], it["d_dur"], d_aln = it["ps"], it["dur"], it["aln"]
            else:                                      # original text, on a timeline that matches the accented one
                dp, dk = dpacks[it["di"]]
                it["d_ps"] = it["orig"]
                it["d_dur"] = back_map_durations(it["orig"], it["ps"], it["origin"], it["dur"],
                                                 _durations_cached(it["orig"], dp, dk, speed))
                d_aln = make_alignment(it["orig"], it["d_dur"])
            it["all_d"] = [predict_f0n(it["d_ps"], dp, d_aln) for dp, _ in dpacks]
            it["f0_d"], it["n_d"] = it["all_d"][it["di"]]
            assert len(it["f0_d"]) == len(it["f0_t"]), "donor and target frames differ"

    # ---- pass 3: normalize each sentence's donor into the target's register ----
    if (pitch or energy) and norm_scope == "text":
        sf_t = f0_stats([it["f0_t"] for it in items]); se_t = en_stats([it["n_t"] for it in items])
        sf_d = [f0_stats([it["all_d"][d][0] for it in items]) for d in range(len(dpacks))]
        se_d = [en_stats([it["all_d"][d][1] for it in items]) for d in range(len(dpacks))]
    for it in items:
        if pitch or energy:
            if norm_scope == "text":
                st_f, st_e, tt_f, tt_e = sf_d[it["di"]], se_d[it["di"]], sf_t, se_t
            else:
                st_f, st_e = f0_stats([it["f0_d"]]), en_stats([it["n_d"]])
                tt_f, tt_e = f0_stats([it["f0_t"]]), en_stats([it["n_t"]])
        it["f0"] = apply_f0(it["f0_d"], st_f, tt_f, f0_mode)       if pitch  else it["f0_t"]
        it["n"]  = apply_energy(it["n_d"], st_e, tt_e, energy_mode) if energy else it["n_t"]

    # ---- pass 4: decode + segments, concatenated like synth_aligned ----
    what = [w for w, on in [("pitch", pitch), ("energy", energy), ("duration", duration)] if on]
    audios, segments, seg_words, seg_altered = [], [], [], []
    T, F0o, F0t, Ne, Nt, chunks, sentences = [], [], [], [], [], [], []
    t0, word_offset = 0.0, 0
    for it in items:
        a = decode(it["ps"], tpack, it["aln"], it["f0"], it["n"])
        segs, widx, altd = align_phonemes(it["ps"], it["dur"], n_samples=len(a), sr=sr, altered_mask=it["mask"])
        segments += [(lab, s + t0, e + t0, info) for lab, s, e, info in segs]
        seg_words += [None if w is None else w + word_offset for w in widx]
        seg_altered += altd
        real = [w for w in widx if w is not None]
        word_offset += (max(real) + 1) if real else 0

        k = len(it["f0"])                                     # prosody curves on the global timeline
        T.append(t0 + np.arange(k) * (len(a) / k) / sr)
        F0o.append(np.where(voiced_mask(it["f0"]), it["f0"], np.nan))
        F0t.append(np.where(voiced_mask(it["f0_t"]), it["f0_t"], np.nan))
        Ne.append(it["n"]); Nt.append(it["n_t"])
        for arr, fill in [(T, t0 + len(a) / sr), (F0o, np.nan), (F0t, np.nan), (Ne, np.nan), (Nt, np.nan)]:
            arr.append(np.array([fill]))                      # NaN gap -> curves break between sentences

        info = {k_: it[k_] for k_ in ("ci", "orig", "ps", "origin", "mask", "fixed", "dur", "dur_own", "dur_t",
                                      "dur_d", "d_dur", "f0", "f0_t", "f0_d", "n", "n_t", "n_d")}
        info.update(chunk=it["ci"], target=_voice_name(target_voice, 0), donor=it["donor"], what=what,
                    donor_text=donor_text)
        chunks.append((a, info))
        sentences.append(dict(chunk=it["ci"], start=t0, end=t0 + len(a) / sr,
                              donor=it["donor"], donor_idx=it["di"]))
        audios.append(a); t0 += len(a) / sr

    pros = dict(t=np.concatenate(T), f0=np.concatenate(F0o), f0_t=np.concatenate(F0t),
                n=np.concatenate(Ne), n_t=np.concatenate(Nt), chunks=chunks, sentences=sentences,
                target=_voice_name(target_voice, 0), donors=names, what=what,
                modes=dict(f0=f0_mode, energy=energy_mode, dur=dur_mode, scope=norm_scope,
                           dur_scope=dur_scope, assign=assign if isinstance(assign, str) else "fixed", seed=seed,
                           donor_text=donor_text, target_timing=target_timing, keep_own=keep_own))
    return np.concatenate(audios), segments, seg_words, seg_altered, pros


def print_assignment(pros):
    from collections import Counter
    for s_ in pros["sentences"]:
        print(f"sentence {s_['chunk']:2d}  {s_['start']:6.2f}–{s_['end']:6.2f} s   prosody from {s_['donor']}")
    print("counts:", dict(Counter(s_["donor"] for s_ in pros["sentences"])))


def show_altered_durations(pros, sentences=(0,)):
    '''Frames of every altered token: the target's own (reading the accented text), the donor's
    (before rate scaling) and what was synthesized. "own" = kept at the target's duration.'''
    for a, info in pros["chunks"]:
        if info["chunk"] not in sentences:
            continue
        ps, mask, origin = info["ps"], info["mask"], info["origin"]
        print(f"sentence {info['chunk']}: donor {info['donor']} reads the {info['donor_text']} text, "
              f"transferred: {', '.join(info['what']) or 'nothing'}")
        print(f"   {'token':>5} {'from':>5} | {'target':>6} {'donor':>6} {'final':>6}")
        for k, j in enumerate([j for j, c in enumerate(ps) if c in VOCAB], start=1):
            if mask is None or not mask[j]:
                continue
            frm = info["orig"][origin[j][0]:origin[j][1]] if origin is not None else "?"
            dd = info["dur_d"][k] if info["dur_d"] is not None else "-"
            print(f"   {ps[j]:>5} {frm:>5} | {info['dur_own'][k]:6d} {dd:>6} {info['dur'][k]:6d}"
                  f"{'   own' if info['fixed'][k] else ''}")
        print()

## Interactive player with a pitch / energy panel
Same player as the prosody notebook: word boxes, token durations, altered phonemes highlighted, plus the pitch/energy panel (one colour per donor; dashed grey = what the target voice would have done on the same timing).

In [ ]:
_PROS_PANEL = r"""
<div id="prosLegend" style="font-size:11.5px;color:#555;margin:8px 0 2px;display:flex;gap:14px;flex-wrap:wrap;align-items:center;">
  <span id="prosDonors"></span>
  <span><span style="display:inline-block;width:22px;border-top:2px dashed #94a3b8;vertical-align:middle"></span> target's own prosody</span>
  <span id="prosInfo" style="color:#777"></span>
</div>
<canvas id="pc" width="1040" height="170" style="width:100%;height:170px;background:#fff;border:1px solid #e5e7eb;border-radius:6px;cursor:pointer"></canvas>
"""

_PROS_JS = r"""
const PROS=__PROS__;
const pc=document.getElementById('pc'), pctx=pc.getContext('2d');
document.getElementById('prosInfo').textContent = PROS.info;
const PAL=['#f59e0b','#10b981','#8b5cf6','#ef4444','#0ea5e9','#ec4899'];
const dcol=i=>PAL[i % PAL.length];
(function(){ const box=document.getElementById('prosDonors');
    PROS.donors.forEach((d,i)=>{ const sp=document.createElement('span'); sp.style.marginRight='12px';
        sp.innerHTML='<span style="display:inline-block;width:22px;border-top:2.5px solid '+dcol(i)
            +';vertical-align:middle"></span> '+d+' <span style="color:#999">×'+PROS.counts[i]+'</span>';
        box.appendChild(sp); }); })();
function drawPros(){
    const W=pc.width, H=pc.height, a=view[0], b=view[1];
    pctx.clearRect(0,0,W,H);
    const TOP=16, H1=Math.round(H*0.66);
    const lo=Math.log(PROS.f0lim[0]), hi=Math.log(PROS.f0lim[1]);
    const X=t=>(t-a)/(b-a)*W;
    const yF=f=>H1-6-(Math.log(f)-lo)/(hi-lo)*(H1-TOP-14);
    const yE=e=>H-4-(e-PROS.nlim[0])/(PROS.nlim[1]-PROS.nlim[0])*(H-H1-14);
    pctx.font='10px system-ui,sans-serif'; pctx.lineWidth=1;
    PROS.sents.forEach(s=>{ if(s.e<a||s.s>b) return;              // donor strip, one block per sentence
        const x0=Math.max(0,X(s.s)), x1=Math.min(W,X(s.e));
        pctx.fillStyle=dcol(s.d)+'33'; pctx.fillRect(x0,0,x1-x0-1,TOP-3);
        pctx.fillStyle=dcol(s.d); pctx.fillRect(x0,TOP-4,x1-x0-1,2);
        pctx.save(); pctx.beginPath(); pctx.rect(x0,0,x1-x0-2,TOP); pctx.clip();
        pctx.fillStyle='#333'; pctx.fillText(PROS.donors[s.d],x0+3,TOP-6); pctx.restore(); });
    PROS.ticks.forEach(f=>{ const y=yF(f); pctx.strokeStyle='#f1f1f1';
        pctx.beginPath(); pctx.moveTo(0,y); pctx.lineTo(W,y); pctx.stroke();
        pctx.fillStyle='#999'; pctx.fillText(f+' Hz',3,y-2); });
    pctx.strokeStyle='#e5e7eb'; pctx.beginPath(); pctx.moveTo(0,H1+2); pctx.lineTo(W,H1+2); pctx.stroke();
    pctx.fillStyle='#999'; pctx.fillText('energy',3,H1+13);
    pctx.strokeStyle='#00000012';                                   // word starts, like the waveform
    groups.forEach(g=>{ if(g.w===null||g.s<a||g.s>b) return; const x=X(g.s);
        pctx.beginPath(); pctx.moveTo(x,TOP); pctx.lineTo(x,H); pctx.stroke(); });
    const t=PROS.t;
    function line(arr, yf, col, dash, lw, t0, t1){
        pctx.strokeStyle=col; pctx.setLineDash(dash); pctx.lineWidth=lw; pctx.beginPath();
        let pen=false;
        for(let i=0;i<t.length;i++){
            const v=arr[i];
            if(v===null || t[i]<Math.max(a,t0) || t[i]>Math.min(b,t1)){ pen=false; continue; }
            const x=X(t[i]), y=yf(v);
            if(pen) pctx.lineTo(x,y); else { pctx.moveTo(x,y); pen=true; }
        }
        pctx.stroke(); pctx.setLineDash([]); pctx.lineWidth=1;
    }
    line(PROS.f0_t, yF, '#94a3b8', [4,3], 1.5, -1, 1e9);
    line(PROS.n_t,  yE, '#94a3b8', [4,3], 1.2, -1, 1e9);
    PROS.sents.forEach(s=>{                                         // synthesized curves in the donor's color
        line(PROS.f0, yF, dcol(s.d), [], 2.2, s.s, s.e);
        line(PROS.n,  yE, dcol(s.d), [], 1.8, s.s, s.e); });
    const cx=X(au.currentTime); pctx.strokeStyle='#f43f5e'; pctx.lineWidth=2;
    pctx.beginPath(); pctx.moveTo(cx,0); pctx.lineTo(cx,H); pctx.stroke(); pctx.lineWidth=1;
}
"""

_IDS = ["pv", "pageTitle", "rulesBox", "statsBox", "statsTotal", "statsBody", "au", "pp", "sp", "spl",
        "tt", "legend", "durNote", "wf", "ph", "pc", "prosLegend", "prosInfo", "prosDonors"]


def _prosody_template():
    """The player template from functions.py with the prosody panel spliced in."""
    t = F._TEMPLATE
    anchors = ['<canvas id="wf"', "let view=[0,DUR];", "function drawWave(){", "\ndrawWave();\n})();"]
    missing = [a for a in anchors if a not in t]
    assert not missing, f"player template changed, anchors not found: {missing}"
    i = t.index('<canvas id="wf"'); j = t.index("</canvas>", i) + len("</canvas>")
    t = t[:j] + _PROS_PANEL + t[j:]
    t = t.replace("let view=[0,DUR];", "let view=[0,DUR];\n" + _PROS_JS, 1)
    t = t.replace("function drawWave(){", "function drawWave(){ drawPros();", 1)
    t = t.replace("\ndrawWave();\n})();", "\npc.onclick=wf.onclick; pc.onwheel=wf.onwheel;\ndrawWave();\n})();", 1)
    return t


def _uniquify_ids(html):
    uid = uuid.uuid4().hex[:8]
    for i in _IDS:
        html = html.replace(f'id="{i}"', f'id="{i}_{uid}"').replace(f"getElementById('{i}')", f"getElementById('{i}_{uid}')")
    return html


def _pros_payload(pros):
    from collections import Counter
    tolist = lambda x, r: [None if not np.isfinite(v) else round(float(v), r) for v in x]
    f0_all = np.concatenate([pros["f0"], pros["f0_t"]]); f0_all = f0_all[np.isfinite(f0_all)]
    lo, hi = np.percentile(f0_all, [1, 99]); lo, hi = lo / 1.15, hi * 1.15
    n_all = np.concatenate([pros["n"], pros["n_t"]]); n_all = n_all[np.isfinite(n_all)]
    ticks = [f for f in (50, 75, 100, 150, 200, 300, 400, 500) if lo < f < hi]
    m = lambda f: np.exp(np.nanmean(np.log(f)))
    md = pros["modes"]
    if pros["what"]:
        info = (" + ".join(pros["what"]) + f" · F0 {md['f0']}, energy {md['energy']}, duration {md['dur']}"
                f" ({md['dur_scope']}), stats per {md['scope']}, assign {md['assign']} (seed {md['seed']})"
                f" · donor reads the {md['donor_text']} text")
    else:
        info = "nothing transferred (native)"
    info += f" · mean F0 {m(pros['f0']):.0f} Hz (target's own {m(pros['f0_t']):.0f} Hz)"
    cnt = Counter(s_["donor_idx"] for s_ in pros["sentences"])
    sents = [dict(s=round(s_["start"], 4), e=round(s_["end"], 4), d=s_["donor_idx"]) for s_ in pros["sentences"]]
    if not pros["what"]:          # native: no donor strip colours
        sents, donors, counts = [], [], []
    else:
        donors, counts = pros["donors"], [cnt.get(i, 0) for i in range(len(pros["donors"]))]
    return dict(t=tolist(pros["t"], 4), f0=tolist(pros["f0"], 1), f0_t=tolist(pros["f0_t"], 1),
                n=tolist(pros["n"], 3), n_t=tolist(pros["n_t"], 3),
                f0lim=[float(lo), float(hi)], nlim=[float(n_all.min()), float(n_all.max())],
                ticks=ticks, info=info, sents=sents, donors=donors, counts=counts)


def audiovisualize_prosody(audio, segments, pros, sr=SR, out_html=None, seg_words=None, text=None,
                           seg_altered=None, title=None, rules=None, stats=None):
    """audiovisualize_interactive + pitch/energy panel. Same arguments, plus `pros` from synth_accent."""
    original = F._TEMPLATE
    try:
        F._TEMPLATE = _prosody_template()
        html = audiovisualize_interactive(audio, segments, sr=sr, out_html=None, seg_words=seg_words,
                                          text=text, seg_altered=seg_altered, title=title,
                                          rules=rules, stats=stats)
    finally:
        F._TEMPLATE = original
    html = _uniquify_ids(html.replace("__PROS__", json.dumps(_pros_payload(pros))))
    if out_html:
        out_dir = os.path.join(os.getcwd(), "output_online"); os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, out_html)
        page = ('<!DOCTYPE html>\n<html><head><meta charset="utf-8">\n'
                '<meta name="viewport" content="width=device-width, initial-scale=1">\n'
                f'<title>{title or "Kokoro"}</title></head>\n<body>\n' + html + '\n</body></html>\n')
        with open(path, "w", encoding="utf-8") as f:
            f.write(page)
        print(f"Writing interactive HTML to: {path}")
    return html

## Settings

In [ ]:
text = (""" Elena Marchetti lavorava da sei anni nell'archivio comunale di Ponteverde, una piccola città costruita lungo un fiume nella pianura del Nord. L'archivio occupava il terzo piano di un vecchio palazzo di mattoni rossi, a pochi passi dalla piazza principale e dal municipio. Il palazzo era stato costruito alla fine dell'Ottocento, e in passato aveva ospitato una scuola e poi alcuni uffici del tribunale. La sala più grande aveva soffitti alti, scaffali di legno scuro e due lunghe finestre che guardavano verso il fiume. In quella sala si trovavano i registri delle nascite, dei matrimoni e delle proprietà, alcuni dei quali avevano più di duecento anni. Il lavoro di Elena consisteva nel catalogare i documenti, nel rispondere alle richieste dei cittadini e nel tenere in ordine le sale. Ogni mattina arrivava alle otto, accendeva le luci, apriva le finestre per qualche minuto e controllava la posta elettronica dell'ufficio. Di solito restava in archivio fino alle sette di sera, perché preferiva lavorare quando il palazzo era tranquillo e silenzioso. Nelle ore del pomeriggio gli altri uffici si svuotavano, e lei poteva concentrarsi sui registri più vecchi senza essere interrotta.
Con lei lavorava anche Paolo Ferretti, un collega più giovane che si occupava soprattutto della digitalizzazione dei documenti. Paolo passava molte ore davanti allo scanner, una grande macchina che fotografava le pagine una per una con molta precisione. Quando aveva tempo, Paolo aiutava Elena anche con le richieste dei cittadini, soprattutto quelle che arrivavano per posta elettronica. Al piano terra, invece, c'era il custode del palazzo, Aldo Rinaldi, che lavorava in quell'edificio da più di trent'anni. Aldo conosceva ogni stanza, ogni porta e ogni scala, e spesso raccontava come era cambiato il palazzo nel corso del tempo. Ogni giovedì Elena passava qualche minuto con Aldo nell'ingresso, per parlare del tempo, dei lavori in città e delle novità del comune.
All'inizio di novembre, una sera, Elena ha sentito per la prima volta un rumore strano che veniva dal soffitto della sala grande. Era una serie di piccoli colpi secchi, simili al suono di qualcuno che batte lentamente con le nocche su una porta di legno. Non assomigliava ai rumori normali di un edificio vecchio, come il vento nelle finestre o i passi delle persone nelle scale. Elena si è alzata dalla scrivania, ha camminato fino al centro della sala e ha guardato il soffitto per qualche momento. Ha pensato per un attimo a una finestra aperta, ma tutte le finestre della sala erano chiuse e il vento non era forte. Non ha visto niente di strano, perché l'intonaco era bianco e liscio come sempre, e le lampade erano ferme al loro posto. Il rumore è durato circa dieci minuti, poi si è fermato all'improvviso, e la sala è tornata silenziosa come sempre. Elena ha pensato che qualcuno stesse lavorando al piano di sopra, ma poi si è ricordata che il terzo piano era l'ultimo. Sopra l'archivio c'era soltanto il tetto, con le travi di legno e uno spazio basso dove nessuno entrava da molti anni.
La sera seguente, verso la stessa ora, il rumore è ricominciato, sempre dallo stesso punto del soffitto, vicino alla finestra. Anche questa volta è durato pochi minuti, e poi è scomparso senza lasciare nessuna traccia e nessuna spiegazione evidente. Elena ha preso un foglio e ha scritto la data, l'ora e una breve descrizione del suono, per non dimenticare i dettagli. Nei giorni successivi il rumore si è ripetuto quasi ogni sera, e qualche volta Elena lo ha sentito anche al mattino presto. Non era un suono forte, ma in una sala così silenziosa era impossibile non notarlo e non chiedersi da dove venisse. Per abitudine, Elena ha messo il foglio nel primo cassetto della scrivania, insieme alle chiavi e agli altri appunti della settimana.
Una mattina Elena ha raccontato tutto a Paolo, mentre preparavano insieme le scatole di documenti da portare allo scanner. Paolo l'ha ascoltata con attenzione e poi ha detto che, secondo lui, nel soffitto potevano esserci dei topi o altri piccoli animali. Ha spiegato che negli edifici vecchi succede spesso, soprattutto in autunno, quando gli animali cercano un posto caldo per l'inverno. Ha raccontato che anche nella casa dei suoi nonni, in campagna, una volta avevano trovato dei topi sotto il tetto. In quel caso, ha detto, gli animali facevano rumore soprattutto di notte, quando la casa era buia e tranquilla. "Se ci sono dei topi sopra la sala, dobbiamo chiamare subito qualcuno, perché potrebbero rovinare anche i documenti", ha detto Paolo. Elena ha pensato che l'idea fosse ragionevole, perché i colpi erano irregolari e potevano sembrare i movimenti di un animale. Tuttavia, le sembrava strano che gli animali facessero rumore sempre alla stessa ora, e quasi sempre per lo stesso numero di minuti. Inoltre, durante il giorno la sala era piena di rumori normali, e forse gli animali si muovevano anche in quelle ore senza farsi sentire. Ha deciso comunque di seguire il consiglio del collega, perché la protezione dei documenti era la cosa più importante per l'archivio. Nel pomeriggio ha scritto una richiesta all'ufficio tecnico del comune, chiedendo un controllo del soffitto e dello spazio sotto il tetto.
Pochi giorni dopo è arrivato un tecnico di una ditta specializzata, con una scala, una torcia potente e una piccola telecamera. La ditta lavorava spesso per il comune, e controllava regolarmente le scuole, le palestre e gli altri edifici pubblici della città. Il tecnico è salito nello spazio sotto il tetto attraverso una botola nel corridoio, che Elena non aveva mai notato prima. Elena e Paolo hanno aspettato nel corridoio, ascoltando i passi del tecnico che si muoveva lentamente sopra le loro teste. Ogni tanto sentivano anche la sua voce, perché parlava al telefono con un collega per descrivere quello che vedeva. Ha passato quasi un'ora a controllare le travi, gli angoli e il pavimento di legno sopra la sala grande dell'archivio. Ha usato la telecamera per guardare anche nei punti più stretti, dove una persona non riusciva a entrare con il corpo. Le immagini apparivano su un piccolo schermo, e il tecnico le osservava con calma prima di passare al punto successivo. Quando è sceso, ha spiegato a Elena e a Paolo che non aveva trovato nessun segno della presenza di animali. Non c'erano nidi, non c'erano escrementi e non c'erano buchi nel legno o nei muri, nemmeno vicino alla finestra. Ha aggiunto che lo spazio era molto pulito e asciutto, e che il tetto sembrava in buone condizioni in ogni punto. Prima di andare via ha lasciato alcune trappole per sicurezza, ma ha detto che probabilmente sarebbero rimaste vuote. Infatti, nelle due settimane seguenti le trappole sono rimaste vuote, mentre il rumore continuava a ripetersi quasi tutte le sere.
Un venerdì sera, mentre usciva dal palazzo, Elena si è fermata a parlare con Aldo, che stava chiudendo la porta principale. Gli ha raccontato la storia del rumore, del tecnico e delle trappole vuote, e gli ha chiesto se avesse mai sentito qualcosa di simile. Aldo ha ascoltato tutta la storia senza interrompere, come faceva sempre quando qualcuno gli parlava di un problema del palazzo. Poi ha detto che in trent'anni non aveva mai sentito rumori strani al terzo piano, né di giorno né di sera. Ha ricordato però che, all'inizio di ottobre, il comune aveva fatto sostituire tutto l'impianto di riscaldamento del palazzo. I lavori erano durati tre settimane, e durante quel periodo l'edificio era stato pieno di operai, di attrezzi e di polvere. Gli operai avevano installato una caldaia nuova e anche dei tubi nuovi, che passavano proprio sopra il soffitto dell'ultimo piano. "Forse non è un caso: il rumore è cominciato poche settimane dopo i lavori sul riscaldamento", ha detto Aldo con calma. Ha aggiunto che, se voleva, poteva darle gli orari del nuovo impianto, perché erano programmati con un timer automatico. Aldo ha spiegato che il timer accendeva e spegneva la caldaia ogni giorno alla stessa ora, senza bisogno di nessun intervento. Elena non aveva mai pensato al riscaldamento, perché i termosifoni della sala funzionavano bene e non facevano nessun rumore. Inoltre, i tubi non si vedevano, perché erano nascosti nello spazio sotto il tetto e nei muri spessi del palazzo. Tornando a casa, però, ha riflettuto sulle parole di Aldo e ha deciso di raccogliere informazioni in modo più preciso.
Dal lunedì seguente Elena ha preparato una tabella semplice, con una colonna per il giorno, una per l'ora e una per la durata. Aveva scelto di usare una tabella perché sapeva, dal suo lavoro in archivio, che i dati scritti sono più affidabili della memoria. Ogni volta che sentiva il rumore, guardava l'orologio e scriveva il momento esatto in cui il suono cominciava e finiva. Ha chiesto anche ad Aldo una copia degli orari del riscaldamento, che erano scritti su un foglio appeso vicino alla caldaia. Il foglio indicava che l'impianto si accendeva alle sei del mattino e si spegneva alle sei e mezza di sera, dal lunedì al venerdì. Nel fine settimana, invece, il riscaldamento restava spento, perché nel palazzo non lavorava nessuno e gli uffici erano chiusi. Nelle prime sere i numeri non le dicevano molto, ma dopo alcuni giorni Elena ha cominciato a vedere una certa regolarità. Per due settimane ha scritto i dati ogni giorno, anche nelle sere in cui il rumore era molto debole e difficile da sentire. Anche Paolo l'ha aiutata, perché nei giorni in cui Elena usciva prima, restava lui in sala e annotava gli orari nella tabella. Il rumore non arrivava mai nel mezzo della giornata, quando la sala era calda e il riscaldamento funzionava senza interruzioni. Una domenica è passata in archivio per prendere un libro, e ha notato che la sala era silenziosa per tutto il pomeriggio. Alla fine di due settimane ha messo la sua tabella accanto al foglio degli orari, e ha confrontato le due serie di numeri.
Il risultato era molto chiaro: il rumore cominciava sempre pochi minuti dopo lo spegnimento del riscaldamento, verso le sei e quaranta. In tutte le sere registrate, la differenza tra lo spegnimento dell'impianto e l'inizio dei colpi era sempre di circa dieci minuti. Al mattino, invece, i colpi arrivavano poco dopo l'accensione, mentre l'acqua calda cominciava a scorrere di nuovo nei tubi. Nei fine settimana, quando l'impianto restava spento, il rumore non si sentiva mai, né al mattino né alla sera. Elena ha mostrato la tabella ad Aldo, che l'ha letta con attenzione e ha confrontato ogni ora con il foglio della caldaia. Aldo ha detto che i numeri non lasciavano dubbi, e ha telefonato subito alla ditta che aveva installato il nuovo impianto. Qualche giorno dopo è arrivato un idraulico, che è salito nello spazio sotto il tetto insieme ad Aldo, con una torcia e alcuni attrezzi. Prima di salire, l'idraulico aveva guardato la tabella di Elena, e aveva detto che quei dati gli avrebbero fatto risparmiare molto tempo. Dopo pochi minuti ha trovato il punto: un tubo di metallo appoggiato direttamente su una vecchia trave di legno, vicino alla finestra. L'idraulico ha chiesto a Elena di restare nella sala grande, e poi ha battuto piano sul tubo con un piccolo attrezzo. Elena ha sentito lo stesso suono secco delle settimane precedenti, e ha confermato che veniva esattamente da quel punto del soffitto. L'idraulico ha spiegato che il metallo si allunga un poco quando diventa caldo e si accorcia di nuovo quando si raffredda. Ogni volta che la temperatura cambiava, il tubo scivolava sulla trave a piccoli scatti, e ogni scatto produceva un colpo secco. "È un fenomeno normale, che succede spesso quando i tubi nuovi vengono montati troppo vicino al legno", ha detto l'idraulico.
Per risolvere il problema, l'idraulico ha sollevato leggermente il tubo e ha inserito tra il metallo e la trave alcuni supporti di gomma. La gomma permetteva al tubo di muoversi liberamente, senza strisciare sul legno e quindi senza produrre rumore durante i cambi di temperatura. Secondo l'idraulico, quei supporti erano economici e semplici da montare, e potevano durare molti anni senza bisogno di manutenzione. Ha controllato anche gli altri tubi sopra il terzo piano, ma erano tutti montati correttamente e lontani dalle travi. L'idraulico ha spiegato che era meglio controllare tutto subito, perché lo stesso problema poteva comparire anche in altri punti. Prima di scendere, Aldo ha scritto la data dell'intervento nel quaderno dove registrava tutte le riparazioni del palazzo. Quella sera Elena è rimasta in archivio fino alle sette, come al solito, e ha ascoltato con attenzione il silenzio della sala. Alle sei e mezza il riscaldamento si è spento, ma dal soffitto non è arrivato nessun colpo, né in quel momento né più tardi. Il giorno dopo ha raccontato a Paolo come erano andate le cose, e insieme hanno tolto le trappole che erano rimaste nel corridoio. Paolo ha osservato che era meglio così, perché i documenti più antichi erano delicati e difficili da sostituire. Elena ha conservato la tabella in una cartella dell'archivio, insieme alla documentazione dei lavori sul riscaldamento del palazzo. Pensava che un giorno qualcun altro avrebbe potuto trovare utile quella informazione, magari durante altri lavori sull'edificio. Poi è tornata ai suoi registri antichi, e le settimane successive sono passate tranquille, tra le richieste dei cittadini e i documenti da catalogare.""")


text = text[:2000]              # shorter text for a demo
target_voice = "im_nicola"      # timbre + segments (native Italian voice)
donor_voices = ["zm_yunjian", "hf_alpha", "jf_alpha"]    # prosody sources, one chosen per sentence
SEED       = 0                  # same seed -> same donor per sentence in every condition
RULE_SEED  = 1                  # reproducible random substitutions

# accent rules, applied in order (see the docstring at the top of functions.py for all kinds)
rules = [{"kind": "substitute_prob", "old": "t", "choices": {"tʲ": 33, "θ": 33, "ɖ": 33}, "same_in_word": True},
         {"kind": "substitute_prob", "old": "k", "choices": {"x": 33, "χ": 33, "kʲ": 33}, "same_in_word": True},
         {"kind": "substitute_prob", "old": "p", "choices": {"β": 33, "ɸ": 33, "pʲ": 33}, "same_in_word": True},
         # exactly N words per vowel, over the whole text, at most ONE vowel change per word
         # (words already changed by the consonant rules above stay candidates unless skip_altered_words=True)
         {"kind": "word_quota",
          "targets": [{"old": "e", "new": "ə", "words": 40, "phonemes": 50},       # <- your v1
                      {"old": "o", "new": "ʊ", "words": 45, "phonemes": 55},       # <- your v2
                      {"old": "i", "new": "ɪ", "words": 50, "phonemes": 60}],      # <- your v3
          "batch": 10,                 # words per vowel per round
          "order": "rotate",           # each round starts at the next vowel ("shuffle": new random order per round)
          "occurrences": "all",        # "all": every e of a chosen word, phonemes ≈ target (word swaps)
                                       # "partial": exactly `phonemes` changes spread over the words
                                       # "one": a single random e per word (phonemes = words)
          "skip_altered_words": False},
         ]

DONOR_TEXT    = "original"      # "original": donor reads the unmanipulated text; "manipulated": the accented one
TARGET_TIMING = "manipulated"   # target's own timing: of the accented text, or "original" (= SAME_TIMING=True)
KEEP_OWN      = ("ʲ",)          # added tokens that keep the target's own duration

## Text → IPA → accented IPA

In [ ]:
ipa_chunks = text_to_ipa_chunks(text)
ipa_chunks = normalize_ipa(ipa_chunks, {"ɪ": "i"})   # BEFORE anything else: native, donor and accented all start from i

accent_chunks, masks, stats, origins = manipulate_with_quota(
    ipa_chunks, rules, return_mask=True, return_stats=True, return_origin=True, seed=RULE_SEED)

for orig, acc in zip(ipa_chunks, accent_chunks):
    print("native  :", orig)
    print("accented:", acc, "\n")
print_stats([r for r in stats if r["kind"] != "word_quota"])

# the words each word_quota target changed
for r in stats:
    if r["kind"] == "word_quota":
        for old, ws in r["words"].items():
            print(f"{old}: {len(ws)} words →", ", ".join(f"{w['word']}({w['new']})" for w in ws[:15]), "…")

assert check_ipa(accent_chunks), "some symbols are not in Kokoro's vocab and would be dropped (see above)"

# everything the accented versions share
ACC = dict(manip_chunks=accent_chunks, masks=masks, origins=origins,
           donor_text=DONOR_TEXT, target_timing=TARGET_TIMING, keep_own=KEEP_OWN,
           assign="balanced", seed=SEED)

## Native: target voice, original text, its own prosody

In [ ]:
audio_nat, segs_nat, words_nat, alt_nat, pros_nat = synth_accent(ipa_chunks, target_voice)

display(HTML(audiovisualize_prosody(audio_nat, segs_nat, pros_nat, seg_words=words_nat, text=text,
                                    seg_altered=alt_nat, title=f"Native: {target_voice}",
                                    out_html="native.html")))

## Segmental accent only: target voice with its own prosody on the accented text

In [ ]:
audio_seg, segs_seg, words_seg, alt_seg, pros_seg = synth_accent(ipa_chunks, target_voice, donor_voices, **ACC)

display(HTML(audiovisualize_prosody(audio_seg, segs_seg, pros_seg, seg_words=words_seg, text=text,
                                    seg_altered=alt_seg, rules=rules, stats=stats,
                                    title=f"Segmental accent: {target_voice}",
                                    out_html="accented_segmental.html")))

## Segmental accent + transferred prosody
Choose what is transferred with the flags. `DONOR_TEXT` (in the settings) decides which text the donor reads; you can also override it here with `donor_text=...`.

In [ ]:
audio_acc, segs_acc, words_acc, alt_acc, pros_acc = synth_accent(
    ipa_chunks, target_voice, donor_voices,
    pitch=True, energy=True, duration=True,
    f0_mode="meanstd", energy_mode="meanstd", dur_mode="rate", norm_scope="text",
    dur_scope="sentence",    # "sentence": every sentence as long as the target's own; "text": only the total
    **ACC)

print_assignment(pros_acc)
display(HTML(audiovisualize_prosody(audio_acc, segs_acc, pros_acc, seg_words=words_acc, text=text,
                                    seg_altered=alt_acc, rules=rules, stats=stats,
                                    title=f"Segmental + prosody ({DONOR_TEXT} text) from {', '.join(donor_voices)}",
                                    out_html=f"accented_segmental_prosody_{DONOR_TEXT}.html")))

### Check the palatalized tokens
For every altered token: the target's own frames, the donor's frames (before rate scaling) and the synthesized ones.
`kʲ`: the `k` has the donor's frames (× the sentence's rate factor with `dur_mode="rate"`), the `ʲ` is marked `own` and keeps the target's frames.

In [ ]:
show_altered_durations(pros_acc, sentences=(0, 1))

## Several conditions in one go
Every combination of what is transferred × which text the donor reads, one player and one wav each. With the same `SEED` every condition uses the same donor for each sentence, and with the same `RULE_SEED` the same substitutions.

In [ ]:
conditions = {
    "pitch":                 dict(pitch=True,  energy=False, duration=False),
    "pitch+energy":          dict(pitch=True,  energy=True,  duration=False),
    "duration":              dict(pitch=False, energy=False, duration=True),
    "pitch+energy+duration": dict(pitch=True,  energy=True,  duration=True),
}
donor_texts = ["original", "manipulated"]

os.makedirs("output_online", exist_ok=True)
results = {}
for dtext in donor_texts:
    for name, flags in conditions.items():
        a, s, w, alt, p = synth_accent(ipa_chunks, target_voice, donor_voices,
                                       **{**ACC, "donor_text": dtext}, **flags)
        results[(name, dtext)] = (a, s, w, alt, p)
        fname = f"seg_{name.replace('+', '_')}_{dtext}"
        sf.write(f"output_online/{fname}.wav", a, SR)
        display(HTML(audiovisualize_prosody(a, s, p, seg_words=w, text=text, seg_altered=alt,
                                            rules=rules, stats=stats,
                                            title=f"Segmental + {name} (donor reads the {dtext} text)",
                                            out_html=f"{fname}.html")))

## Save the audio

In [ ]:
os.makedirs("output_online", exist_ok=True)
sf.write("output_online/native.wav", audio_nat, SR)
sf.write("output_online/accented_segmental.wav", audio_seg, SR)
sf.write(f"output_online/accented_segmental_prosody_{DONOR_TEXT}.wav", audio_acc, SR)
print(f"native               {len(audio_nat)/SR:7.3f} s")
print(f"segmental            {len(audio_seg)/SR:7.3f} s")
print(f"segmental + prosody  {len(audio_acc)/SR:7.3f} s")

# per-sentence check (frames of 25 ms)
for (a_s, i_s), (a_a, i_a) in zip(pros_seg["chunks"], pros_acc["chunks"]):
    print(f"sentence {i_s['chunk']}: segmental {sum(i_s['dur']):4d} fr / {len(a_s)/SR:6.3f} s   "
          f"+ prosody {sum(i_a['dur']):4d} fr / {len(a_a)/SR:6.3f} s")